In [1]:
# Import libraries and other configurations
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sns.set_theme(style="whitegrid")

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 121

In [2]:
# Load the cleaned dataset
df = pd.read_csv(
    "../data/processed/cleaned_donor_data.csv",
    dtype={"donor_postal_code": "str"}
)

df.columns.tolist()

['donor_unique_id',
 'donor_postal_code',
 'donor_age',
 'gender_identity',
 'is_member_flag',
 'is_alumnus_flag',
 'is_parent_flag',
 'has_involvement_flag',
 'preferred_address_type',
 'has_email_flag',
 'consecutive_donor_years',
 'last_fiscal_year_donation',
 'donation_2_fiscal_years_ago',
 'donation_3_fiscal_years_ago',
 'donation_4_fiscal_years_ago',
 'donation_5_fiscal_years_ago',
 'current_fiscal_year_donation',
 'cumulative_donation_amount',
 'donor_indicator_flag']

In [3]:
# Verify dataset loaded correctly
print(f"Dataset Shape: {df.shape}")
df.sample(10, random_state=RANDOM_STATE)

Dataset Shape: (34403, 19)


,donor_unique_id,donor_postal_code,donor_age,gender_identity,is_member_flag,is_alumnus_flag,is_parent_flag,has_involvement_flag,preferred_address_type,has_email_flag,consecutive_donor_years,last_fiscal_year_donation,donation_2_fiscal_years_ago,donation_3_fiscal_years_ago,donation_4_fiscal_years_ago,donation_5_fiscal_years_ago,current_fiscal_year_donation,cumulative_donation_amount,donor_indicator_flag
33267,33371,42301.0,46,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
26595,26683,45620.0,42,Female,0,0,0,0,Home,0,1,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
16653,16704,33427.0,28,Female,0,0,0,0,Home,0,0,0.00,120.00,0.00,0.00,0.00,0.00,120.00,1
21428,21498,54131.0,33,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1
16873,16924,90265.0,33,Male,0,1,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
19919,19982,54459.0,32,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
15478,15527,64686.0,42,Male,0,0,0,0,Home,0,8,0.00,0.00,10.00,0.00,0.00,0.00,"9,680.00",1
591,596,12067.0,42,Female,0,0,1,0,Home,0,0,0.00,100.00,1.00,0.00,0.00,0.00,101.00,1
1474,1483,90265.0,42,Female,0,1,0,0,Home,1,0,0.00,0.00,0.00,0.00,0.00,0.00,479.00,1
22846,22922,45856.0,36,Female,0,0,0,0,Home,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0


In [4]:
# Verify dataset data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34403 entries, 0 to 34402
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   donor_unique_id               34403 non-null  int64  
 1   donor_postal_code             34312 non-null  str    
 2   donor_age                     34403 non-null  int64  
 3   gender_identity               33912 non-null  str    
 4   is_member_flag                34403 non-null  int64  
 5   is_alumnus_flag               34403 non-null  int64  
 6   is_parent_flag                34403 non-null  int64  
 7   has_involvement_flag          34403 non-null  int64  
 8   preferred_address_type        30370 non-null  str    
 9   has_email_flag                34403 non-null  int64  
 10  consecutive_donor_years       34403 non-null  int64  
 11  last_fiscal_year_donation     34403 non-null  float64
 12  donation_2_fiscal_years_ago   34403 non-null  float64
 13  donation_3_f

In [5]:
# Data types summary
dtype_summary = (
    df.dtypes
      .astype(str)
      .value_counts()
      .reset_index()
      .style.hide(axis="index")
)

dtype_summary.columns = ["Data Type", "Count"]
dtype_summary

index,count
int64,9
float64,7
str,3


In [6]:
# Verify dataset missing values
missing_summary = (
    df.isna()
      .sum()
      .to_frame("Missing Count")
)

missing_summary["Missing %"] = (
    missing_summary["Missing Count"] / len(df) * 100
)

missing_summary = (
    missing_summary
      .sort_values("Missing Count", ascending=False)
)

missing_summary

,Missing Count,Missing %
preferred_address_type,4033,11.72
gender_identity,491,1.43
donor_postal_code,91,0.26
donor_unique_id,0,0.00
last_fiscal_year_donation,0,0.00
cumulative_donation_amount,0,0.00
current_fiscal_year_donation,0,0.00
donation_5_fiscal_years_ago,0,0.00
donation_4_fiscal_years_ago,0,0.00
donation_3_fiscal_years_ago,0,0.00


## Dataset Overview
The finalized cleaned analytical dataset was successfully loaded and verified for feature engineering. The dataset contains 34,403 donor records and 19 features, consistent with the dataset used throughout Phase 2. Data types and missing values match expectations from the data cleaning process, and all required libraries were successfully imported. The only change made was explicitly defining `donor_postal_code` as a string. This was made because it was interpreting postal codes as a float and not a string.

In [7]:
# Define target-related column names
TARGET_SOURCE_COLUMN = "current_fiscal_year_donation"
TARGET_COLUMN = "target_current_fiscal_year_donor_flag"

# Validate the target source before creating the binary target
assert pd.api.types.is_numeric_dtype(df[TARGET_SOURCE_COLUMN]), (
    f"{TARGET_SOURCE_COLUMN} must contain numeric values."
)

# Check for missing values
missing_target_values = df[TARGET_SOURCE_COLUMN].isna().sum()
print(f"Missing target-source values: {missing_target_values:,}")

assert missing_target_values == 0, (
    f"{TARGET_SOURCE_COLUMN} contains "
    f"{missing_target_values:,} missing values."
)

# Check for negative numbers
negative_target_values = (df[TARGET_SOURCE_COLUMN] < 0).sum()
print(f"Negative target-source values: {negative_target_values:,}")

assert negative_target_values == 0, (
    f"{TARGET_SOURCE_COLUMN} contains "
    f"{negative_target_values:,} negative values."
)

Missing target-source values: 0
Negative target-source values: 0


In [8]:
# Create the binary prediction target
df[TARGET_COLUMN] = (
    df[TARGET_SOURCE_COLUMN] > 0
).astype("int8")

print(f"Target column created: {TARGET_COLUMN}")

Target column created: target_current_fiscal_year_donor_flag


In [9]:
# Summarize the target distribution
target_summary = (
    df[TARGET_COLUMN]
    .value_counts()
    .sort_index()
    .rename_axis(TARGET_COLUMN)
    .reset_index(name="count")
)

target_summary["percentage"] = (
    target_summary["count"] / len(df) * 100
).round(2)

target_summary["target_label"] = target_summary[TARGET_COLUMN].map(
    {
        0: "Did not donate",
        1: "Donated",
    }
)

target_summary = target_summary[
    [
        TARGET_COLUMN,
        "target_label",
        "count",
        "percentage",
    ]
]

display(target_summary.style.hide(axis="index").format({"percentage": "{:.2f}%", "count": "{:,}"}))

target_current_fiscal_year_donor_flag,target_label,count,percentage
0,Did not donate,"32,499",94.47%
1,Donated,"1,904",5.53%


In [10]:
# Confirm that the target contains only binary values and matches its source column
expected_target_values = {0, 1}
actual_target_values = set(df[TARGET_COLUMN].unique())

assert actual_target_values == expected_target_values, (
    f"Unexpected target values found: {actual_target_values}"
)

# Confirm that the target matches its source column
target_mismatch_count = (
    df[TARGET_COLUMN]
    != (df[TARGET_SOURCE_COLUMN] > 0).astype("int8")
).sum()

assert target_mismatch_count == 0, (
    f"Found {target_mismatch_count:,} target-definition mismatches."
)

print("Target validation passed.")

Target validation passed.


## Prediction Target and Time Window

The classification target is `target_current_fiscal_year_donor_flag`, which identifies whether an individual donated during the current fiscal year.

The target is derived from `current_fiscal_year_donation`:

- `0`: The individual donated $0 during the current fiscal year
  
- `1`: The individual donated more than $0 during the current fiscal year

The target distribution shows that 1,904 individuals, or 5.53% of the dataset, donated during the current fiscal year. The remaining 32,499 individuals, or 94.47%, did not donate. This indicates a substantial class imbalance that will need to be considered during model evaluation and training.

The intended prediction setup uses the five completed fiscal years before the current fiscal year as the historical observation window. The current fiscal year serves as the outcome period.

The model will therefore use information available before the beginning of the current fiscal year to predict whether an individual donates during that fiscal year.

The original `current_fiscal_year_donation` column will remain in the working dataset temporarily for validation, but it must not be included in the predictor matrix because it directly determines the target.

Until additional documentation is available:

* `cumulative_donation_amount` will not be used directly because it may include current-fiscal-year donations
* `donor_indicator_flag` will not be used as a predictor until its calculation timing and logic are confirmed
* `consecutive_donor_years` will not be used until it is confirmed that it excludes current-fiscal-year activity
* Demographic and engagement fields will be treated as provisionally available
* The exact fiscal-year dates and dataset extraction date should be confirmed before the final model is deployed

The timing and leakage status of all source variables will be evaluated during the formal leakage audit.
